In [ ]:
from keras import layers, models
import keras
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
data = np.load('./datasets/reuters.npz',allow_pickle=True)

In [ ]:
X = data['x']
y = data['y']

ratio =  int(np.ceil(len(X) * .9))


X_train, X_test = X[:ratio],X[ratio:]
y_train, y_test = y[:ratio],y[ratio:]



In [ ]:
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension), dtype="float32")
    for i, seq in enumerate(sequences):
        seq = [w for w in seq if w < dimension]   # keep only the top words
        results[i, seq] = 1.0
    return results

x_train = vectorize_sequences(X_train)
x_test = vectorize_sequences(X_test)

In [ ]:
def one_hot_encode(labels, num_classes=46):
    results = np.zeros((len(labels), num_classes))
    for i, label in enumerate(labels):
        results[i, label] = 1.0
    return results

y_train = one_hot_encode(y_train)
y_test = one_hot_encode(y_test)

In [ ]:
model = models.Sequential()

model.add(layers.Input(shape=(10000, )))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(46, activation='softmax'))

In [ ]:
model.compile(optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
x_validation = x_train[:1000]
x_train = x_train[1000: ]

y_validation = y_train[:1000]
y_train = y_train[1000: ]

In [ ]:
history = model.fit(x_train,
y_train,
epochs=30,
batch_size=512,
validation_data=(x_validation, y_validation))

In [ ]:
hist = history.history

training_acc = hist['accuracy']
validation_acc = hist['val_accuracy']

training_loss = hist['loss']
validation_loss = hist['val_loss']

In [ ]:
epoch = range(1, len(training_loss) + 1)
plt.plot(epoch, training_loss, c='lime', ls='-', label='Training Loss',markersize=10)
plt.plot(epoch, validation_loss, c='purple', ls='dotted', label='Validation Loss',markersize=10)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
plt.clf()
epoch = range(1, len(training_acc) + 1)
plt.plot(epoch, training_acc, c='lime', ls='-', label='Training Accuracy',markersize=10,alpha=0.8)
plt.plot(epoch, validation_acc, c='purple', ls='dashed', label='Validation Accuracy',markersize=10, alpha=0.8)
plt.legend()
plt.show()

In [ ]:
results = model.evaluate(x_test, y_test)
results

In [ ]:
predict = model.predict(x_test)
print(predict[0])

### `In this project, I learned that Dense layers work best with fixed-length inputs in which each position has the same meaning across all samples. Raw word IDs break this requirement: they are arbitrary dictionary indices, not quantities, so the network can’t extract meaning from them. Instead, I used multi-hot vectorization, which turns each article into a vector where position k indicates whether word k appears. Every input neuron then has a consistent meaning across all samples. What multi-hot encoding does. Each article becomes a vector of length 10,000 (the vocabulary size) with 1 where a word appears and 0 elsewhere. Rare words beyond the vocabulary are dropped.`